In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import heapq
import seaborn as sns

from environment.environment import GraphWorldMFG_MultiGroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer
from solver.solver import solve_multigroup, GraphMFG_OMD_EdgeSolver_MultiGroup
from visualization.visualizationh import plot_heatmap, compute_exploitability_multigroup, plot_losses, plot_losses_line

In [2]:
import os
from pathlib import Path
import yaml
import numpy as np
import torch

# 1. Configuration Loader
def load_config(config_path="config.yaml"):
    """Load configuration safely from a YAML file relative to working directory."""
    notebook_dir = Path(os.getcwd())
    config_file = notebook_dir / config_path
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    return config

# 2. Graph Environment Factory Pattern
def create_graph_mfg_from_config(config):
    """Factory function initializing the Graph MFG environment directly from your config."""
    device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")

    trainer_cfg = config["trainer"]
    
    # Extract Graph Configuration
    graph_cfg = config["graph"]
    num_nodes = graph_cfg["num_nodes"]
    
    # CRITICAL: Extract the list of lists matrix and convert to a PyTorch Tensor
    raw_matrix = graph_cfg["adjacency_matrix"]
    adjacency_matrix = torch.tensor(raw_matrix, dtype=torch.float32, device=device)
    
    # Process Group Data (Sinks and Sources are flat integers here, not grid tuples)
    groups = []
    for g in config["groups"]:
        groups.append({
            "source": int(g["source"]),
            "sink": int(g["sink"]),
            "mass": float(g["mass"])
        })
    
    solver_cfg = config["solver"]
    waiting_time = solver_cfg.get("waiting_time", 0)
    # Instantiate the Graph World Environment
    # (Matches your 'GraphWorldMFG_MultiGroup' class structure)
    env = GraphWorldMFG_MultiGroup(
        num_nodes=num_nodes,
        groups=groups,
        adjacency_matrix=adjacency_matrix,
        H = solver_cfg.get("H", None),
        device=device
    )
    
    # Create solvers for each group
    
    
    solvers = [
        GraphMFG_OMD_EdgeSolver_MultiGroup(
            env=env,
            group_idx=k,
            eta=solver_cfg["eta"],
            tau=solver_cfg["tau"],
            T=solver_cfg["T"],
            alpha=solver_cfg["alpha"],
            H=solver_cfg.get("H", None)
        )
        for k in range(env.K)
    ]

    trainer = GraphEdgeMFG_Trainer(env, solvers, leader_lr=trainer_cfg["leader_lr"])
    
    return env, solvers, trainer, config

In [3]:
config = load_config("config_graph.yaml")

env, solvers, trainer, config = create_graph_mfg_from_config(config)

# Training loop
theta_leader = torch.zeros((env.N, env.N), device=env.device)  # Initialize leader's strategy

Target Device: cpu


c:\Uzh\thesis\msc_code\thesis\environment\environment.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.A = torch.tensor(adjacency_matrix, dtype=torch.float32, device=device)


In [23]:
flows, final_flows, policies, W_cong_history, zeta_history = solve_multigroup(env, solvers, T=50, W_max=15, theta_leader=theta_leader)

In [25]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0894e-07, 1.0000e+00, 0.0000e+00],
        [1.6817e-15, 4.4362e-13, 5.0000e-01, 5.0000e-01],
        [3.3635e-15, 3.3645e-15, 1.0405e-21, 1.0000e+00],
        [3.3635e-15, 3.3645e-15, 1.0405e-21, 1.0000e+00],
        [3.3635e-15, 3.3645e-15, 1.0405e-21, 1.0000e+00],
        [3.3635e-15, 3.3645e-15, 1.0405e-21, 1.0000e+00],
        [3.3635e-15, 3.3645e-15, 1.0405e-21, 1.0000e+00],
        [1.6817e-15, 1.6823e-15, 1.6817e-15, 1.0000e+00],
        [1.1359e-29, 1.0437e-21, 3.3635e-15, 1.0000e+00],
        [1.1359e-29, 1.0437e-21, 3.3635e-15, 1.0000e+00],
        [1.1359e-29, 1.0437e-21, 3.3635e-15, 1.0000e+00],
        [1.135

In [26]:
flows[0,:,:,0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.0893e-07, 5.0000e-01, 0.0000e+00],
        [0.0000e+00, 4.4194e-13, 5.0000e-01, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 2.5000e-01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 7.5000e-01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [1.6817e-15, 1.6823e-15, 1.0405e-21, 1.0000e+00],
        [1.6817e-15, 1.6823e-15, 1.4884e-27, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.000

In [21]:
theta_leader = torch.tensor([[5.0000, 5.0000, 0.0000, 1.2216],
        [5.0000, 5.0000, 5.0000, 0.0000],
        [5.0000, 5.0000, 4.0590, 0.0000],
        [5.0000, 4.7202, 5.0000, 5.0000]])

In [15]:
flows, final_flows, policies, W_cong_history = env.simulate_forward_with_policy(policies, theta_leader=theta_leader, W_max=15)

In [16]:
final_flows

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0000, 0.9238, 0.0762, 0.0000],
        [0.0037, 0.9275, 0.0381, 0.0307],
        [0.0074, 0.9312, 0.0000, 0.0614],
        [0.0074, 0.9312, 0.0000, 0.0614],
        [0.0074, 0.9312, 0.0000, 0.0614],
        [0.0414, 0.5794, 0.0341, 0.3451],
        [0.0968, 0.0074, 0.0895, 0.8063],
        [0.0935, 0.0074, 0.0898, 0.8093],
        [0.0902, 0.0074, 0.0902, 0.8123],
        [0.0902, 0.0074, 0.0902, 0.8123],
        [0.0902, 0.0074, 0.0902, 0.8123],
        [0.0594, 0.0443, 0.0566, 0.8397],
        [0.0094, 0.0991, 0.0071, 0.8844],
        [0.0094, 0.0957, 0.0073, 0.8876]])

In [17]:
flows[0,:,:,0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 3.8082e-02, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 3.8082e-02, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 3.0706e-02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 6.1413e-02],
        [0.0000e+00, 3.5182e-01, 0.0000e+00, 6.1413e-02],
        [0.0000e+00, 5.7202e-01, 0.0000e+00, 6.1413e-02],
        [3.6878e-03, 3.6878e-03, 0.0000e+00, 6.1413e-02],
        [3.6878e-03, 3.6878e-03, 0.0000e+00, 6.1413e-02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 6.1413e-02],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 6.1413e-02],
        [3.4069e-02, 0.0000e+00, 3.4069e-02, 2.0325e-01],
        [5.539

In [18]:
a = trainer.compute_social_loss(flows, W_cong_history)
a

tensor(17.0422)

In [19]:
loss = 0
for i in range(env.H):
    cost = flows[0, i, 3, 0] - 1
    loss = loss + cost
loss

tensor(-17.0422)

In [13]:
print("Starting Leader Infrastructure Optimization...")
print("-" * 50)
flows_previous = flows.detach()  # Detach to avoid backprop through the entire history
W_cong_history = W_cong_history.detach()  # Detach to avoid backprop
losses = []

epochs = 250
for epoch in range(1, epochs + 1):
    
    # Execute one optimization step
    social_loss, flows_new, W_cong_history_new, theta_leader_new = trainer.train_step(flows_previous, W_cong_history, epoch)
    
    # Pass the newly generated flows as the historical footprint for the next epoch
    flows_previous = flows_new.detach() 
    W_cong_history = W_cong_history_new.detach()
    theta_leader = theta_leader_new.detach()

    # Optional: Recompute or update W_cong_history based on flows_new if required, 
    # otherwise it continues to adapt based on the internal solve_multigroup loop.
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:02d}/{epochs} | Social Loss (Travel Time): {social_loss:.4f}")
    
    losses.append(social_loss)

print("-" * 50)
print("Training Complete!")

Starting Leader Infrastructure Optimization...
--------------------------------------------------


c:\Uzh\thesis\msc_code\thesis\solver\solver.py:210: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  v_node = torch.cat([v_choice, torch.tensor(v_waiting, device=self.env.device)], dim=0)


Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
theta _ leader out is tensor([[2.3516, 2.6422, 2.6253, 2.3814],
        [2.5100, 2.5293, 2.6325, 2.5700],
        [2.4898, 2.5763, 2.3762, 2.5346],
        [2.6249, 2.4977, 2.5177, 2.4538]])
Epoch 01/250 | Social Loss (Travel Time): 17.0813
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
theta _ leader out is tensor([[2.1268, 2.1550, 2.9532, 2.2504],
        [3.3101, 2.2948, 2.6671, 2.6319],
        [2.8424, 3.0240, 2.1464, 2.2311],
        [2.7592, 1.5944, 2.3333, 2.5275]])
Epoch 02/250 | Social Loss (Travel Time): 17.2318
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
theta _ leader out is tensor([[1.1138, 4.6259, 2.1240, 1.0583],
        [3.5976, 1.3146, 4.6875, 0.6862],
        [4.7409, 4.5474, 0.9627, 0.0402],
        [4.1378, 0.9257, 0.8776, 3.4000]])
Epoch 03/250 | Social Loss (Travel Time): 12.2221
Adjoint vector a_T for group 0 has shape: torch.Size([20, 4, 4])
theta _ leader ou

KeyboardInterrupt: 

In [ ]:
theta_leader * env.A

tensor([[-0.0000e+00, -1.0000e+01, -2.0820e-23, -0.0000e+00],
        [-1.0000e+01, -0.0000e+00, -1.0000e+01, -0.0000e+00],
        [-1.0000e+01, -1.0000e+01, -0.0000e+00, -5.7592e-33],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00]])

In [ ]:
flows, final_flows, policies, W_cong_history = env.simulate_forward_with_policy(policies, theta_leader=theta_leader, W_max=8)

In [ ]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [0.0000e+00, 9.9004e-01, 9.9590e-03, 0.0000e+00],
        [7.4129e-03, 0.0000e+00, 1.7372e-02, 9.7522e-01],
        [7.4129e-03, 0.0000e+00, 1.7372e-02, 9.7522e-01],
        [5.5504e-05, 7.4684e-03, 9.9590e-03, 9.8252e-01],
        [5.5504e-05, 7.4684e-03, 9.9590e-03, 9.8252e-01],
        [9.3203e-05, 9.2788e-05, 5.0354e-03, 9.9478e-01],
        [1.3049e-04, 1.3007e-04, 5.5919e-05, 9.9968e-01],
        [3.8397e-05, 1.3091e-04, 6.9475e-07, 9.9983e-01],
        [1.3926e-06, 1.3091e-04, 9.7391e-07, 9.9987e-01],
        [9.8535e-07, 3.8402e-05, 9.8015e-07, 9.9996e-01],
        [9.8744e-07, 1.3999e-06, 9.8015e-07, 1.0000e+00],
        [2.9488e-07, 9.9269e-07, 2.8754e-07, 1.0000e+00],
        [1.7821e-08, 9.9478e-07, 1.0482e-08, 1.0000e+00],
        [9.5857e-09, 2.9703e-07, 7.4328e-09, 1.0000e+00],
        [7.5269e-09, 1.7899e-08, 7.4484e-09, 1.0000e+00],
        [2.279

In [ ]:
flows[0,:,3,0]

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0073, 0.0073, 0.4998,
        0.9923, 0.9924, 0.9924, 0.9961, 0.9997, 0.9998, 0.9999, 0.9999, 1.0000,
        1.0000, 1.0000])

In [ ]:
loss = 0
for i in range(env.H):
    cost = flows[0, i, 3, 0] - 1
    loss = loss + cost

In [ ]:
loss

tensor(-8.5132)